# Session 15: Classification: yes or no questions

Encode the loan categories, fit a logistic regression and a small tree, compare them at one threshold, then choose a threshold from the cost of each mistake. Your scikit-learn numbers will be close to the slides but will differ a little.

**SOLUTION VERSION. Tutor only.** Run the cells from the top. Each cell that starts with '# Your turn' is yours to complete. Run it, then run the check cell after it: the check tells you if you are right and what to fix.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, zipfile, urllib.request
DATA_ZIP_URL = ""   # your tutor may give you a link; leave empty to upload data.zip instead
if not os.path.exists("orders.csv"):
    if DATA_ZIP_URL:
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

## Task 1. Load and prepare the loans
Read `loans.csv`. Keep ages from 18 to 80, fill missing incomes with the median, and add a `burden` column: loan amount times 1.15, divided by the term, divided by income.

In [ ]:
import pandas as pd
import numpy as np

loans = pd.read_csv('loans.csv')
loans = loans[loans.age.between(18, 80)].copy()
med = loans.monthly_income_ugx.median()
loans['monthly_income_ugx'] = loans.monthly_income_ugx.fillna(med)
loans['burden'] = (loans.loan_amount_ugx * 1.15 / loans.term_months
                   / loans.monthly_income_ugx)
print(loans.shape)

In [ ]:
assert loans.shape == (7998, 19), 'Keep ages 18 to 80 (7998 loans) and add one column, burden'
assert loans.monthly_income_ugx.isna().sum() == 0, 'Fill the missing incomes'
assert 'burden' in loans.columns, 'Add the burden column'
print('Task 1 correct')

## Task 2. Encode the categories
Drop `loan_id`, `disbursed_date`, `district`, `collection_calls` and `default` from `loans` to make `features`. Then use `pd.get_dummies(features, dtype=int)` to make `X`, and print its shape.

In [ ]:
features = loans.drop(columns=['loan_id', 'disbursed_date', 'district',
                               'collection_calls', 'default'])
X = pd.get_dummies(features, dtype=int)
print(X.shape)

In [ ]:
assert X.shape == (7998, 26), 'Drop the five columns, then expect 26 columns after encoding'
assert 'occupation_Teacher' in X.columns, 'get_dummies makes one column per occupation. Pass the text columns through it'
print('Task 2 correct')

## Task 3. Fit a logistic regression
Write the three money columns in millions, split 75 and 25 with `random_state=42`, fit a `LogisticRegression(max_iter=1000)` and store the chance of default for each test loan in `p`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

money = ['monthly_income_ugx', 'loan_amount_ugx', 'savings_balance_ugx']
X[money] = X[money] / 1_000_000
y = loans['default']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42)
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
p = model.predict_proba(X_test)[:, 1]
print(p.round(3)[:5])

In [ ]:
assert len(p) == len(X_test), 'One probability for each test loan'
assert 0.08 <= p.mean() <= 0.17, 'The average chance should be near 0.125. Take the column for class 1, the chance of default'
print('Task 3 correct')

## Task 4. Fit a decision tree
Fit a `DecisionTreeClassifier(max_depth=3, random_state=42)` on the same training loans. Print its rules with `export_text`, then draw it with `plot_tree`.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
import matplotlib.pyplot as plt

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)
print(export_text(tree, feature_names=list(X.columns)))
plt.figure(figsize=(14, 6))
plot_tree(tree, feature_names=list(X.columns), filled=True, fontsize=8)
plt.show()

In [ ]:
assert tree.get_depth() <= 3, 'Use max_depth=3'
assert tree.get_n_leaves() >= 2, 'Fit the tree on X_train and y_train'
print('Task 4 correct')

## Task 5. Compare the two models
Both models give a probability for every test loan. Store the tree's chances for the test loans in `p_tree`. Flag the loans with a probability of 0.2 or more. For each model, store the number of real defaults it caught in `caught_lr` and `caught_tree`, and the number of loans it flagged in `flagged_lr` and `flagged_tree`.

In [ ]:
p_tree = tree.predict_proba(X_test)[:, 1]
real = (y_test == 1).to_numpy()
caught_lr = int(((p >= 0.2) & real).sum())
caught_tree = int(((p_tree >= 0.2) & real).sum())
flagged_lr = int((p >= 0.2).sum())
flagged_tree = int((p_tree >= 0.2).sum())
print('defaults in the test loans:', int(real.sum()))
print('logistic regression caught', caught_lr, 'from', flagged_lr, 'flags')
print('tree caught', caught_tree, 'from', flagged_tree, 'flags')

In [ ]:
real_n = int(y_test.sum())
assert flagged_lr == int((p >= 0.2).sum()), 'flagged_lr counts the loans with p of 0.2 or more'
assert flagged_tree == int((p_tree >= 0.2).sum()), 'flagged_tree counts the loans with p_tree of 0.2 or more'
assert 0 < caught_lr <= flagged_lr, 'Caught cannot exceed flagged. Count flagged loans that really defaulted'
assert 0 <= caught_tree <= flagged_tree, 'Caught cannot exceed flagged. Count flagged loans that really defaulted'
assert caught_lr <= real_n and caught_tree <= real_n, 'Caught cannot exceed the real defaults in the test loans'
print('Task 5 correct')

## Task 6. Which occupation defaults most?
Compute the default rate for each `occupation`, sorted from highest to lowest, in `rates`. Store the name of the highest in `worst`.

In [ ]:
rates = loans.groupby('occupation').default.mean().sort_values(ascending=False)
worst = rates.index[0]
print(rates.round(3))
print(worst)

In [ ]:
assert worst == 'Boda rider', 'Sort the rates from highest to lowest, then take the first name of the index'
assert abs(rates.iloc[0] - 0.198) < 0.005, 'rates should hold the mean of default for each occupation, highest first'
print('Task 6 correct')

## Task 7. Choose a threshold from costs
The starter cell gives ten test loans with a predicted probability `prob` and a true outcome `truth` (1 means default). A missed default costs 1,000,000 UGX. A refused good loan costs 100,000 UGX. These are invented teaching costs. For each threshold in `thresholds`, add the total cost to the list `costs`, and store the cheapest threshold in `best_t`.

In [ ]:
prob = np.array([0.04, 0.08, 0.10, 0.15, 0.22, 0.28, 0.35, 0.45, 0.60, 0.75])
truth = np.array([0, 0, 0, 1, 0, 1, 0, 1, 1, 1])
thresholds = [0.05, 0.1, 0.2, 0.3, 0.5]
costs = []
for t in thresholds:
    flagged = prob >= t
    missed = ((~flagged) & (truth == 1)).sum()
    refused = (flagged & (truth == 0)).sum()
    costs.append(missed * 1_000_000 + refused * 100_000)
best_t = thresholds[int(np.argmin(costs))]
print(costs)
print(best_t)

In [ ]:
assert len(costs) == 5, 'Add one cost for each of the five thresholds'
assert costs[2] == 1_200_000, 'Cost = missed * 1,000,000 + refused * 100,000. The total at threshold 0.2 should be 1,200,000'
assert best_t == 0.1, 'best_t is the threshold with the smallest cost. Try np.argmin(costs)'
print('Task 7 correct')

## Task 8. Reflect
Write two sentences. First, which model caught more defaults at the 0.2 threshold, and how many loans did it flag to do it? Second, why is the cheapest threshold well below 0.5 when a missed default costs ten times a refused loan? Then change the cost of a refused loan to 500,000 in Task 7 and see how `best_t` moves.